# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You'll see how to examine the Croissant schema, review available record sets, extract data using the unique `@id` values, and perform some exploratory data analysis (EDA).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets by @id
record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}  | name: {rs.get('name', 'N/A')}")

# Display all fields and columns for each record set
overview = {}
for rs in record_sets:
    rs_id = rs["@id"]
    fields = rs.get("field", [])
    if isinstance(fields, dict):
        fields = [fields]
    columns = rs.get("column", [])
    if isinstance(columns, dict):
        columns = [columns]
    print(f"\nRecord Set: {rs_id}")
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict):
            fid = field.get("@id", field)
            print(f"    - @id: {fid}")
        elif isinstance(field, str):
            print(f"    - @id: {field}")
    print("  Columns:")
    for column in columns:
        if isinstance(column, dict):
            cid = column.get("@id", column)
            print(f"    - @id: {cid}")
        elif isinstance(column, str):
            print(f"    - @id: {column}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from main record set(s):
record_sets = dataset.record_sets

# Pick the first record set if available (adjust as needed)
if len(record_sets) > 0:
    record_set_ids = [rs["@id"] for rs in record_sets]
else:
    # If no record sets are defined in the Croissant, manually specify if known
    record_set_ids = []

dataframes = {}
for record_set_id in record_set_ids:
    # Use the record_set @id for extraction
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

if record_set_ids:
    primary_set = record_set_ids[0]
    print(f"Columns in record set {primary_set}:")
    print(dataframes[primary_set].columns.tolist())
    dataframes[primary_set].head()
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, automatically pick a numeric field and a group field if present
if record_set_ids:
    primary_set = record_set_ids[0]
    df = dataframes[primary_set]
    
    print(f"Available columns: {list(df.columns)}")
    
    # Try to pick a numeric column automatically for analysis
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field '{numeric_field_id}' for analysis.")
        threshold = df[numeric_field_id].quantile(0.9)  # Example threshold: 90th percentile
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())
    else:
        print("No numeric fields found for analysis.")
    
    # Try to pick a group field
    group_fields = [c for c in df.columns if c.lower().startswith(('sex', 'gender', 'site', 'location', 'anatomical', 'type', 'status'))]
    if group_fields and numeric_fields:
        group_field = group_fields[0]
        print(f"Grouping by '{group_field}':")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No categorical/group fields found for grouping.")
else:
    print("No record sets or dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if record_set_ids and numeric_fields:
    plt.figure(figsize=(8, 5))
    df = dataframes[primary_set]
    plt.hist(df[numeric_field_id].dropna(), bins=15, color='skyblue', edgecolor='k', alpha=0.7)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()
    
    if group_fields:
        import seaborn as sns
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load and explore the FAIR² dataset described by a Croissant schema using the `mlcroissant` Python library. We:
- Loaded and described the dataset via its schema
- Listed available record sets and fields by their unique `@id`
- Extracted data to a pandas DataFrame
- Performed elementary EDA, including filtering and normalization of numeric values
- Visualized distributions and group differences

For more detailed domain analysis, see the dataset documentation and explore additional fields by their `@id`s.